In [4]:
WORLD_CUP_2026 = [
    "Algeria",
    "Argentina",
    "Australia",
    "Austria",
    "Belgium",
    "Bosnia and Herzegovina",
    "Brazil",
    "Canada",
    "Cape Verde",
    "Colombia",
    "Croatia",
    "Curacao",
    "Czech Republic",
    "DR Congo",
    "Ecuador",
    "Egypt",
    "England",
    "France",
    "Germany",
    "Ghana",
    "Haiti",
    "Iran",
    "Iraq",
    "Ivory Coast",
    "Japan",
    "Jordan",
    "Mexico",
    "Morocco",
    "Netherlands",
    "New Zealand",
    "Norway",
    "Panama",
    "Paraguay",
    "Portugal",
    "Qatar",
    "Saudi Arabia",
    "Scotland",
    "Senegal",
    "South Africa",
    "South Korea",
    "Spain",
    "Sweden",
    "Switzerland",
    "Tunisia",
    "Turkey",
    "United States",
    "Uruguay",
    "Uzbekistan",
]

In [5]:
import pandas as pd

matches_check = pd.read_csv("../../data/cleaned_matches.csv")
competitions = sorted(matches_check['competition'].unique())

comp_categories = {
    'World Cup': [],
    'Continental Cup': [],
    'Nations League': [],
    'Qualifiers': [],
    'Friendly': []
}

for comp in competitions:
    c_lower = comp.lower()
    if 'fifa world cup' in c_lower and 'qualification' not in c_lower:
        comp_categories['World Cup'].append(comp)
    elif any(x in c_lower for x in ['euro', 'copa américa', 'copa america', 'african cup of nations', 'afc asian cup', 'gold cup', 'confederations cup', 'oceania nations cup']) and 'qualification' not in c_lower:
        comp_categories['Continental Cup'].append(comp)
    elif 'nations league' in c_lower and 'qualification' not in c_lower:
        comp_categories['Nations League'].append(comp)
    elif 'qualification' in c_lower or 'qualifying' in c_lower:
        comp_categories['Qualifiers'].append(comp)
    else:
        comp_categories['Friendly'].append(comp)

print("Competition Categories:")
print(f"\nWorld Cup (K=60): {len(comp_categories['World Cup'])}")
print(f"Continental Cup (K=40): {len(comp_categories['Continental Cup'])}")
print(f"Nations League (K=35): {len(comp_categories['Nations League'])}")
print(f"Qualifiers (K=30): {len(comp_categories['Qualifiers'])}")
print(f"Friendly (K=20): {len(comp_categories['Friendly'])}")

Competition Categories:

World Cup (K=60): 1
Continental Cup (K=40): 8
Nations League (K=35): 2
Qualifiers (K=30): 17
Friendly (K=20): 99


In [6]:
import pandas as pd
from collections import defaultdict

# =========================
# CONFIG
# =========================

INITIAL_ELO = 1500
HOME_ADVANTAGE = 30

FORM_WINDOW = 10

# =========================
# COMPETITION K-FACTORS
# =========================

def get_k_factor(competition: str):
    c = str(competition).lower()
    
    if "fifa world cup" in c and "qualification" not in c:
        return 60
    
    continental_tournaments = [
        "uefa euro", "copa américa", "copa america", 
        "african cup of nations", "afc asian cup", 
        "gold cup", "confederations cup", "oceania nations cup"
    ]
    if any(x in c for x in continental_tournaments) and "qualification" not in c:
        return 40
    
    if "nations league" in c and "qualification" not in c:
        return 35
    
    if "qualification" in c or "qualifying" in c:
        return 30
    
    return 20


# =========================
# LOAD DATA
# =========================

matches = pd.read_csv("../../data/cleaned_matches.csv")

matches["date"] = pd.to_datetime(matches["date"])
matches = matches.sort_values("date").reset_index(drop=True)

# =========================
# STORAGE
# =========================

ratings = defaultdict(lambda: INITIAL_ELO)

match_history = defaultdict(list)
elo_history = []

feature_rows = []

all_teams = set(matches["home_team"]).union(set(matches["away_team"]))

# =========================
# HELPER: FORM CALC
# =========================

def compute_form(team, history):
    recent = history[-FORM_WINDOW:]

    if not recent:
        return 0.5

    points = 0

    for r in recent:
        points += r

    return round(points / (3 * len(recent)), 3)


# =========================
# PROCESS MATCHES
# =========================

for i, match in matches.iterrows():

    home = match["home_team"]
    away = match["away_team"]

    home_goals = match["home_goals"]
    away_goals = match["away_goals"]
    competition = match["competition"]
    neutral = match["neutral_site"]

    home_elo_before = ratings[home]
    away_elo_before = ratings[away]

    home_form_before = compute_form(home, match_history[home])
    away_form_before = compute_form(away, match_history[away])

    home_adv = 0 if neutral else HOME_ADVANTAGE

    expected_home = 1 / (
        1 + 10 ** ((away_elo_before - home_elo_before - home_adv) / 400)
    )
    expected_away = 1 - expected_home

    if home_goals > away_goals:
        actual_home, actual_away = 1, 0
        home_points, away_points = 3, 0

    elif home_goals < away_goals:
        actual_home, actual_away = 0, 1
        home_points, away_points = 0, 3

    else:
        actual_home, actual_away = 0.5, 0.5
        home_points, away_points = 1, 1

    K = get_k_factor(competition)

    goal_diff = abs(home_goals - away_goals)

    if goal_diff <= 1:
        margin = 1.0
    elif goal_diff == 2:
        margin = 1.3
    else:
        margin = min(1.6 + 0.1 * (goal_diff - 3), 2.0)

    ratings[home] += K * margin * (actual_home - expected_home)
    ratings[away] += K * margin * (actual_away - expected_away)

    match_history[home].append(home_points)
    match_history[away].append(away_points)

    elo_history.append({
        "match_id": i,
        "date": match["date"],
        "home_team": home,
        "away_team": away,
        "home_elo_before": home_elo_before,
        "away_elo_before": away_elo_before,
        "home_elo_after": ratings[home],
        "away_elo_after": ratings[away],
    })

    feature_rows.append({
        "match_id": i,
        "date": match["date"],
        "home_team": home,
        "away_team": away,
        "competition": competition,
        "neutral_site": neutral,

        "home_elo_before": home_elo_before,
        "away_elo_before": away_elo_before,

        "home_form_before": home_form_before,
        "away_form_before": away_form_before,

        "home_goals": home_goals,
        "away_goals": away_goals,

        "result": (
            "home_win" if home_goals > away_goals else
            "away_win" if home_goals < away_goals else
            "draw"
        )
    })


# =========================
# BUILD TEAMS TABLE
# =========================

teams_df = pd.DataFrame([
    {
        "team": team,
        "current_elo": round(ratings[team], 2),
        "current_form": compute_form(team, match_history[team]),
        "confederation": ""
    }
    for team in sorted(all_teams)
])

teams_df.to_csv("../../data/teams_with_elo.csv", index=False)

pd.DataFrame(feature_rows).to_csv(
    "../../data/matches_with_features.csv",
    index=False
)

pd.DataFrame(elo_history).to_csv(
    "../../data/elo_history.csv",
    index=False
)

wc_2026_teams = teams_df[teams_df["team"].isin(WORLD_CUP_2026)].copy()
wc_2026_teams = wc_2026_teams.sort_values("current_elo", ascending=False).reset_index(drop=True)
wc_2026_teams["rank"] = range(1, len(wc_2026_teams) + 1)
wc_2026_teams = wc_2026_teams[["rank", "team", "current_elo", "current_form"]]
wc_2026_teams.to_csv("../../data/world_cup_2026_elo.csv", index=False)

wc_matches = pd.DataFrame(feature_rows)
wc_matches_filtered = wc_matches[
    (wc_matches["home_team"].isin(WORLD_CUP_2026)) | 
    (wc_matches["away_team"].isin(WORLD_CUP_2026))
].copy()
wc_matches_filtered.to_csv("../../data/wc_2026_matches_features.csv", index=False)

wc_elo_history = pd.DataFrame(elo_history)
wc_elo_history_filtered = wc_elo_history[
    (wc_elo_history["home_team"].isin(WORLD_CUP_2026)) | 
    (wc_elo_history["away_team"].isin(WORLD_CUP_2026))
].copy()
wc_elo_history_filtered.to_csv("../../data/wc_2026_elo_history.csv", index=False)

print(f"\nTotal matches processed: {len(matches)}")
print(f"Total teams: {len(all_teams)}")
print(f"WC 2026 teams: {len(wc_2026_teams)}")
print(f"WC 2026 matches: {len(wc_matches_filtered)}")
print(f"\nTop 10 WC 2026 teams by ELO:")
print(wc_2026_teams.head(10).to_string(index=False))


Total matches processed: 25316
Total teams: 321
WC 2026 teams: 47
WC 2026 matches: 11945

Top 10 WC 2026 teams by ELO:
 rank      team  current_elo  current_form
    1     Spain      2093.01         0.800
    2 Argentina      2088.13         0.833
    3    France      2023.12         0.833
    4    Brazil      1983.08         0.633
    5   England      1977.21         0.733
    6  Colombia      1969.51         0.733
    7  Portugal      1959.13         0.700
    8   Morocco      1941.07         0.800
    9     Japan      1937.73         0.767
   10   Germany      1937.57         0.900
